In [1]:
!pip install langgraph langchain-groq -q

In [2]:
from google.colab import userdata
import os

from langchain_groq import ChatGroq


os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant",
    temperature=0
)


print("Groq connected successfully")

Groq connected successfully


In [3]:
from typing import TypedDict, List


class ChatState(TypedDict):

    messages: List[str]
    name: str
    preferences: str

In [4]:
def chatbot(state: ChatState) -> dict:

    user_msg = state["messages"][-1]

    name = state.get("name", "")
    prefs = state.get("preferences", "")


    if "my name is" in user_msg.lower():

        name = (
            user_msg.lower()
            .split("my name is")[-1]
            .strip()
            .title()
        )


    if "i like" in user_msg.lower():

        prefs = (
            user_msg.lower()
            .split("i like")[-1]
            .strip()
        )


    context = (
        f"The user's name is {name}. "
        f"They like {prefs}."
    )


    reply = llm.invoke(
        context +
        " Reply naturally to: " +
        user_msg
    ).content


    return {

        "messages": state["messages"] + [reply],

        "name": name,

        "preferences": prefs

    }

In [5]:
from langgraph.graph import StateGraph, START, END


builder = StateGraph(ChatState)


builder.add_node(
    "chatbot",
    chatbot
)


builder.add_edge(
    START,
    "chatbot"
)


builder.add_edge(
    "chatbot",
    END
)


graph = builder.compile()


print("Graph created successfully")

Graph created successfully


In [ ]:
state = {
    "messages": [],
    "name": "",
    "preferences": ""
}


while True:

    user = input("You: ")


    if user.lower() == "quit":
        break


    state["messages"].append(user)


    state = graph.invoke(state)


    print(
        "Bot:",
        state["messages"][-1]
    )

You:  My name is Chandana and I like AI
Bot: Nice to meet you, Chandana. AI is a fascinating field, isn't it? What aspect of AI are you particularly interested in - machine learning, natural language processing, or something else?
You: What is my name and what do I like?
Bot: Nice to meet you, Chandana And I Like Ai. It's great to know that you like AI.
